In [0]:
import pandas as pd
import requests
import datetime
import time
import random
from io import StringIO
from pyspark.sql.functions import current_timestamp

HEX_PATH     = "/Volumes/sima_rakshak_catalog/bronze_layer/raw/done.csv"
SILVER_TABLE = "sima_rakshak_catalog.silver.hex_unified_live"

def get_tactical_data(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, "longitude": lon,
        "current": ["temperature_2m", "wind_speed_10m", "precipitation"],
        "forecast_days": 1
    }
    try:
        r = requests.get(url, params=params, timeout=10).json()
        cur = r.get("current", {})
        elev = r.get("elevation", 0)
        temp = cur.get("temperature_2m", 0)
        
        
        
        # 1. Visibility: Injected based on elevation (Higher = lower vis due to atmospheric haze/fog)
        
        vis_base = 10000 
        visibility = max(200, vis_base - (elev * 1.5) + random.randint(-800, 800))
        
        # 2. Cloud Pct: Injected to be higher at mountain peaks
        cloud_pct = min(100, (elev / 40) + random.randint(10, 25))
        
        # 3. Denseness: Injected based on ecological tree-line logic
        if elev < 2100:
            denseness = random.uniform(0.7, 0.9)  # Dense Forest
        elif elev < 3100:
            denseness = random.uniform(0.3, 0.6)  # Shrub/Scrub
        else:
            denseness = random.uniform(0.0, 0.2)  # Barren / Alpine Rock
            
        # 4. Snow Depth: Higher + Colder = Accumulated Snow
        snow_depth = max(0, (elev - 2400) / 75) if temp < 3 else 0
        
        # 5. Roughness: Simulated surface complexity
        inclination = float(abs(elev % 20)) 
        roughness = (inclination * 0.12) + random.uniform(0.1, 0.4)

        return {
            "elevation_m": float(elev),
            "inclination_deg": inclination,
            "roughness_idx": float(roughness),
            "denseness": float(denseness),
            "temp_c": float(temp),
            "wind_ms": float(cur.get("wind_speed_10m", 0) / 3.6),
            "precip_mm": float(cur.get("precipitation", 0)),
            "visibility_m": float(visibility),
            "cloud_pct": float(cloud_pct),
            "snow_depth_cm": float(snow_depth)
        }
    except:
        # Fallback to zeroed schema if API fails
        return {k: 0.0 for k in ["elevation_m", "inclination_deg", "roughness_idx", "denseness", "temp_c", "wind_ms", "precip_mm", "visibility_m", "cloud_pct", "snow_depth_cm"]}

# --- EXECUTION ---
print("🚀 Sima Rakshak: Processing Tactical Terrain Layer...")
df = pd.read_csv(HEX_PATH).head(75)

final_records = []
for idx, row in df.iterrows():
    lat, lon = row['center_lat'], row['center_lon']
    data = get_tactical_data(lat, lon)
    
    final_records.append({
        "hex_id": row['hex_id'],
        "lat": lat,
        "lon": lon,
        **data,
        "observation_time": datetime.datetime.now()
    })

    if idx % 25 == 0:
        print(f"  Processed {idx}/75 sectors...")

# --- WRITE TO DELTA (Fresh Schema) ---
spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
spark.sql(f"""
CREATE TABLE {SILVER_TABLE} (
    hex_id STRING, 
    lat DOUBLE, 
    lon DOUBLE, 
    elevation_m DOUBLE, 
    inclination_deg DOUBLE, 
    roughness_idx DOUBLE, 
    denseness DOUBLE, 
    temp_c DOUBLE, 
    wind_ms DOUBLE, 
    precip_mm DOUBLE, 
    visibility_m DOUBLE, 
    cloud_pct DOUBLE, 
    snow_depth_cm DOUBLE, 
    observation_time TIMESTAMP, 
    processed_at TIMESTAMP
) USING DELTA
""")

pdf = pd.DataFrame(final_records).fillna(0)
silver_df = spark.createDataFrame(pdf).withColumn("processed_at", current_timestamp())

silver_df.write.format("delta").mode("append").saveAsTable(SILVER_TABLE)

print(f"✅ SUCCESS: {SILVER_TABLE} updated with terrain-aware data injection.")
display(spark.table(SILVER_TABLE))

🚀 Sima Rakshak: Processing Tactical Terrain Layer...
  Processed 0/75 sectors...
  Processed 25/75 sectors...
  Processed 50/75 sectors...
✅ SUCCESS: sima_rakshak_catalog.silver.hex_unified_live updated with terrain-aware data injection.


hex_id,lat,lon,elevation_m,inclination_deg,roughness_idx,denseness,temp_c,wind_ms,precip_mm,visibility_m,cloud_pct,snow_depth_cm,observation_time,processed_at
HEX_0000,32.52862230401004,74.87194850928567,281.0,1.0,0.23190312952151804,0.8332440696273506,29.4,3.944444444444444,0.0,10285.5,17.025,0.0,2026-04-18T05:04:32.037Z,2026-04-18T05:05:08.023Z
HEX_0001,32.51563204806692,74.86452236041224,282.0,2.0,0.5806549588841793,0.7801288880544629,29.4,3.944444444444444,0.0,9333.0,26.05,0.0,2026-04-18T05:04:32.480Z,2026-04-18T05:05:08.023Z
HEX_0002,32.502641332714035,74.85709834226839,277.0,17.0,2.397256934967401,0.7235716874769464,29.5,3.944444444444444,0.0,9344.5,31.925,0.0,2026-04-18T05:04:32.926Z,2026-04-18T05:05:08.023Z
HEX_0003,32.503026017644146,74.87013549001699,282.0,2.0,0.564533520605624,0.8639936580294625,29.4,3.944444444444444,0.0,8977.0,22.05,0.0,2026-04-18T05:04:33.368Z,2026-04-18T05:05:08.023Z
HEX_0004,32.507151525255104,74.85381285939958,276.0,16.0,2.2064811302790672,0.8350550096619855,29.5,3.944444444444444,0.0,9199.0,29.9,0.0,2026-04-18T05:04:33.810Z,2026-04-18T05:05:08.023Z
HEX_0005,32.51127490813735,74.83748874606528,274.0,14.0,1.9168580615084903,0.7121190269202857,29.5,3.944444444444444,0.0,10144.0,30.85,0.0,2026-04-18T05:04:34.259Z,2026-04-18T05:05:08.023Z
HEX_0006,32.500487520400185,74.80018354330345,269.0,9.0,1.3516622049901792,0.7039551045172018,28.9,4.111111111111112,0.0,8819.5,25.725,0.0,2026-04-18T05:04:34.701Z,2026-04-18T05:05:08.023Z
HEX_0007,32.4868805272618,74.79450240384018,270.0,10.0,1.5267589063532805,0.7980617352107464,28.9,4.111111111111112,0.0,9586.0,31.75,0.0,2026-04-18T05:04:35.147Z,2026-04-18T05:05:08.023Z
HEX_0008,32.495564417844164,74.79629547803465,270.0,10.0,1.5067505985131322,0.8722533183390262,28.9,4.111111111111112,0.0,9799.0,17.75,0.0,2026-04-18T05:04:35.590Z,2026-04-18T05:05:08.023Z
HEX_0009,32.49493432198676,74.77928039481506,267.0,7.0,1.2176958696835711,0.8366146154483923,29.0,4.111111111111112,0.0,9652.5,28.675,0.0,2026-04-18T05:04:36.037Z,2026-04-18T05:05:08.023Z
